In [1]:
import os
import sys
sys.path.append(os.path.abspath("../../.."))

In [2]:
import pandas as pd
from classes.trading.actionPredictionTrading import ActionPredictionTrading

# Load the dataset
csv_path = "../../datasets/b3_dados/processed/acoes_concat.csv"
df = pd.read_csv(csv_path, parse_dates=['Date'])

test_start_date = '2019-01-01'

# Filtra o conjunto de teste
test_data = df[df['Date'] >= test_start_date]

In [29]:


# List of stocks and paths to models and scalers
stocks_models_scalers = {
  
    "ELET3": {
        "model_path": "../../saved_models/linear_regression/ELET3_model_v1.0.pkl",
        "scaler_x_path": "../../saved_models/linear_regression/ELET3_scaler_X_v1.0.pkl"
    },
    "EQTL3": {
        "model_path": "../../saved_models/linear_regression/EQTL3_model_v1.0.pkl",
        "scaler_x_path": "../../saved_models/linear_regression/EQTL3_scaler_X_v1.0.pkl"
    },
    "CMIG4": {
        "model_path": "../../saved_models/linear_regression/CMIG4_model_v1.0.pkl",
        "scaler_x_path": "../../saved_models/linear_regression/CMIG4_scaler_X_v1.0.pkl"
    },
    "BRAP3": {
        "model_path": "../../saved_models/linear_regression/BRAP3_model_v1.0.pkl",
        "scaler_x_path": "../../saved_models/linear_regression/BRAP3_scaler_X_v1.0.pkl"
    },


}


In [30]:
# Iterate over each stock
# Definir períodos conforme Moura (2023)
periods = {
    "pre_pandemia": ("2019-01-01", "2019-12-31"),
    "durante_pandemia": ("2020-01-01", "2021-08-31"),
    "pos_pandemia": ("2021-09-01", "2022-09-30")  # ou df['Date'].max() se quiser ir até o fim dos dados
}

# Loop por período
results = {}

for period_name, (start_date, end_date) in periods.items():
    print(f"\n===== Analisando período: {period_name.replace('_', ' ').title()} =====")
    period_data = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

    for stock, paths in stocks_models_scalers.items():
        print(f"\nAnalyzing stock: {stock}")
        analysis = ActionPredictionTrading(
            period_data, stock, model_path=paths["model_path"]
        )
        analysis.load_model()
        if "scaler_x_path" in paths:
            analysis.load_scaler(paths["scaler_x_path"])
        analysis.generate_predictions()

        result_no_stop = analysis.simulate_trading(stop_loss=False, stop_type='percent', stop_value=0.02)
        result_with_stop = analysis.simulate_trading(stop_loss=True, stop_type='percent', stop_value=0.02)
        result_bh = analysis.simulate_buy_and_hold()

        results[(stock, period_name)] = {
            'no_stop_loss': result_no_stop,
            'with_stop_loss': result_with_stop,
            'buy_and_hold': result_bh
        }

# Exibir os resultados
for (stock, period_name), result in results.items():
    print(f"\nResults for {stock} - {period_name.replace('_', ' ').title()}:")
    print(f"No Stop Loss: {result['no_stop_loss']}")
    print(f"With Stop Loss: {result['with_stop_loss']}")
    print(f"Buy and Hold: {result['buy_and_hold']}")




===== Analisando período: Pre Pandemia =====

Analyzing stock: ELET3
Model loaded and validated from ../../saved_models/linear_regression/ELET3_model_v1.0.pkl
Scaler loaded and validated for window=3

Analyzing stock: EQTL3
Model loaded and validated from ../../saved_models/linear_regression/EQTL3_model_v1.0.pkl
Scaler loaded and validated for window=3

Analyzing stock: CMIG4
Model loaded and validated from ../../saved_models/linear_regression/CMIG4_model_v1.0.pkl
Scaler loaded and validated for window=3

Analyzing stock: BRAP3
Model loaded and validated from ../../saved_models/linear_regression/BRAP3_model_v1.0.pkl
Scaler loaded and validated for window=3

===== Analisando período: Durante Pandemia =====

Analyzing stock: ELET3
Model loaded and validated from ../../saved_models/linear_regression/ELET3_model_v1.0.pkl
Scaler loaded and validated for window=3

Analyzing stock: EQTL3
Model loaded and validated from ../../saved_models/linear_regression/EQTL3_model_v1.0.pkl
Scaler loaded a

In [31]:
# Converter resultados em estrutura plana para DataFrame
flat_results = []
for (stock, period_name), metrics in results.items():
    row = {
        'stock': stock,
        'period': period_name,
        'modelo': 'RegressaoLinear',
        'retorno_no_stop': metrics['no_stop_loss']['total_return'],
        'acerto_no_stop': metrics['no_stop_loss']['hit_rate'],
        'sharpe_no_stop': metrics['no_stop_loss']['sharpe_ratio'],
        'drawdown_no_stop': metrics['no_stop_loss']['max_drawdown'],
        'capital_no_stop': metrics['no_stop_loss']['final_capital'],
        'retorno_stop': metrics['with_stop_loss']['total_return'],
        'acerto_stop': metrics['with_stop_loss']['hit_rate'],
        'sharpe_stop': metrics['with_stop_loss']['sharpe_ratio'],
        'drawdown_stop': metrics['with_stop_loss']['max_drawdown'],
        'capital_stop': metrics['with_stop_loss']['final_capital'],
        'retorno_bh': metrics['buy_and_hold']['total_return'],
        'capital_bh': metrics['buy_and_hold']['final_capital'],
        'dias_bh': metrics['buy_and_hold']['days_held']
    }
    flat_results.append(row)

df_results = pd.DataFrame(flat_results)

# Salvar com criação de pasta
csv_path = "../../datasets/trading/trading_results.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    df_combined = pd.concat([df_existing, df_results], ignore_index=True)
    df_combined.drop_duplicates(subset=['stock', 'period', 'modelo'], keep='last', inplace=True)
    df_combined.to_csv(csv_path, index=False)
    print(f"\n Resultados adicionados em: {csv_path}")
else:
    df_results.to_csv(csv_path, index=False)
    print(f"\n Arquivo criado com resultados: {csv_path}")




 Resultados adicionados em: ../../datasets/trading/trading_results.csv


In [32]:
trading_results = pd.read_csv(csv_path)
trading_results


,stock,period,modelo,retorno_no_stop,acerto_no_stop,sharpe_no_stop,drawdown_no_stop,capital_no_stop,retorno_stop,acerto_stop,sharpe_stop,drawdown_stop,capital_stop,retorno_bh,capital_bh,dias_bh
0,ITUB4,pre_pandemia,RegressaoLinear,-0.001228,0.504098,-0.011581,0.005051,99877.172470,0.004090,0.504098,0.042209,0.003563,100408.998173,0.001228,100122.827530,245
1,ITUB4,durante_pandemia,RegressaoLinear,0.003081,0.511002,0.013339,0.011014,100308.065033,0.031633,0.511002,0.168949,0.004113,103163.266880,-0.003081,99691.934967,410
2,ITUB4,pos_pandemia,RegressaoLinear,0.001097,0.507519,0.009222,0.007172,100109.703255,0.007566,0.507519,0.069037,0.004819,100756.555561,-0.001097,99890.296745,267
3,PETR3,pre_pandemia,RegressaoLinear,-0.001320,0.467213,-0.030999,0.002277,99867.989922,0.001553,0.467213,0.041050,0.001454,100155.328989,0.001878,100187.797070,248
4,PETR3,durante_pandemia,RegressaoLinear,0.000559,0.474328,0.004979,0.006789,100055.882549,0.014172,0.474328,0.148018,0.001902,101417.155749,-0.000630,99936.985970,413
5,PETR3,pos_pandemia,RegressaoLinear,-0.009211,0.447368,-0.096780,0.013228,99078.861141,0.002162,0.447368,0.027935,0.004289,100216.183311,0.009068,100906.808281,270
6,VALE3,pre_pandemia,RegressaoLinear,-0.001750,0.475410,-0.009123,0.008331,99825.036621,0.012439,0.475410,0.071638,0.004259,101243.912067,0.002262,100226.158714,248
7,VALE3,durante_pandemia,RegressaoLinear,-0.035606,0.484108,-0.069250,0.060833,96439.368820,0.033579,0.484108,0.082489,0.015947,103357.870316,0.035395,103539.482117,413
8,VALE3,pos_pandemia,RegressaoLinear,0.010354,0.522556,0.025337,0.032908,101035.411453,0.057358,0.522556,0.172715,0.011160,105735.843140,-0.011628,98837.223434,270
9,BBAS3,pre_pandemia,RegressaoLinear,-0.002826,0.491803,-0.035432,0.003848,99717.365265,0.003849,0.491803,0.056181,0.002818,100384.911139,0.002786,100278.622437,248
